# Federated Leukaemia Detection — C-NMC Dataset

**One-time setup:** Upload `C-NMC_Leukemia.zip` to your Google Drive once. Then this notebook mounts Drive and reads it directly — no re-uploading ever again.

### Before running:
1. Go to **drive.google.com** → drag and drop `C-NMC_Leukemia.zip` anywhere in My Drive
2. Set Colab runtime to **T4 GPU** (Runtime → Change runtime type)
3. Run all cells top to bottom

In [ ]:
# CELL 1 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted!')

In [ ]:
# CELL 2 — Find zip in Drive and extract
import os, subprocess

result = subprocess.run(
    ['find', '/content/drive/MyDrive', '-name', 'C-NMC_Leukemia.zip'],
    capture_output=True, text=True
)
zip_path = result.stdout.strip().split('\n')[0]

if not zip_path:
    print('ERROR: C-NMC_Leukemia.zip not found in Google Drive.')
    print('Please upload it to drive.google.com and re-run this cell.')
else:
    print(f'Found zip: {zip_path}')
    if os.path.exists('/content/leukemia_data'):
        print('Already extracted — skipping.')
    else:
        print('Extracting (takes ~2-3 min)...')
        os.makedirs('/content/leukemia_data', exist_ok=True)
        os.system(f'unzip -q "{zip_path}" -d /content/leukemia_data')
        print('Done!')

In [ ]:
# CELL 3 — Auto-detect DATA_ROOT for C-NMC folder structure
import os

DATA_ROOT = None
for root, dirs, files in os.walk('/content/leukemia_data'):
    lower = [d.lower() for d in dirs]
    if 'all' in lower and 'hem' in lower:
        DATA_ROOT = root
        break

if DATA_ROOT is None:
    # Fallback: find any dir with 2+ image-filled subfolders
    for root, dirs, _ in os.walk('/content/leukemia_data'):
        img_dirs = []
        for d in dirs:
            p = os.path.join(root, d)
            imgs = [f for f in os.listdir(p) if f.lower().endswith(('.jpg','.jpeg','.png','.bmp'))]
            if len(imgs) > 5:
                img_dirs.append(d)
        if len(img_dirs) >= 2:
            DATA_ROOT = root
            break

if DATA_ROOT:
    print(f'DATA_ROOT: {DATA_ROOT}')
    for d in os.listdir(DATA_ROOT):
        p = os.path.join(DATA_ROOT, d)
        if os.path.isdir(p):
            n = len([f for f in os.listdir(p) if f.lower().endswith(('.jpg','.jpeg','.png'))])
            print(f'  {d}: {n} images')
else:
    print('Could not find data — printing structure:')
    for root, dirs, _ in os.walk('/content/leukemia_data'):
        depth = root.count(os.sep) - '/content/leukemia_data'.count(os.sep)
        if depth < 5:
            print('  '*depth + os.path.basename(root) + '/')

In [ ]:
# CELL 4 — Imports and config
import copy, random
import numpy as np
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader, Subset, random_split
from torchvision import datasets, transforms
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                              precision_score, recall_score)

SEED=42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

NUM_CLIENTS  = 5
NUM_ROUNDS   = 10
LOCAL_EPOCHS = 3
BATCH_SIZE   = 64
LR           = 0.001
IMG_SIZE     = 64

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
if DEVICE.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# CELL 5 — Load dataset
train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.5,0.5,0.5],[0.5,0.5,0.5])
])
test_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5,0.5,0.5],[0.5,0.5,0.5])
])

full = datasets.ImageFolder(DATA_ROOT, transform=train_tf)
CLASS_NAMES = full.classes
print(f'Classes : {CLASS_NAMES}')
print(f'Total   : {len(full)} images')

train_size = int(0.8 * len(full))
test_size  = len(full) - train_size
train_ds, test_ds = random_split(full, [train_size, test_size],
                                  generator=torch.Generator().manual_seed(SEED))
test_ds.dataset.transform = test_tf
print(f'Train   : {train_size}  |  Test : {test_size}')

In [ ]:
# CELL 6 — Define model and federated functions
class LightweightCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3,16,3,padding=1), nn.BatchNorm2d(16), nn.ReLU(True), nn.MaxPool2d(2),
            nn.Conv2d(16,32,3,padding=1), nn.BatchNorm2d(32), nn.ReLU(True), nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,padding=1), nn.BatchNorm2d(64), nn.ReLU(True), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Dropout(0.4), nn.Linear(64*8*8,128), nn.ReLU(True),
            nn.Dropout(0.3), nn.Linear(128,2)
        )
    def forward(self, x):
        return self.classifier(self.features(x).view(x.size(0),-1))

def partition_noniid(dataset, n, seed=42):
    np.random.seed(seed)
    labels = np.array([dataset.dataset.targets[i] for i in dataset.indices])
    ns = n*2; ss = len(dataset)//ns
    si = np.argsort(labels)
    shards = [si[i*ss:(i+1)*ss] for i in range(ns)]
    order  = np.random.permutation(ns)
    return [np.concatenate([shards[order[c*2]], shards[order[c*2+1]]]).tolist() for c in range(n)]

def local_train(model, ds, idx, epochs, lr, bs, dev):
    loader = DataLoader(Subset(ds,idx), batch_size=bs, shuffle=True,
                        num_workers=2, pin_memory=True)
    m   = copy.deepcopy(model).to(dev)
    opt = optim.Adam(m.parameters(), lr=lr)
    crit = nn.CrossEntropyLoss()
    m.train()
    for _ in range(epochs):
        for imgs, labs in loader:
            imgs, labs = imgs.to(dev, non_blocking=True), labs.to(dev, non_blocking=True)
            opt.zero_grad()
            crit(m(imgs), labs).backward()
            opt.step()
    return m.state_dict(), len(idx)

def fed_avg(cw_list, sizes):
    total = sum(sizes); avg = copy.deepcopy(cw_list[0])
    for k in avg:
        avg[k] = sum((n/total)*w[k].float() for w,n in zip(cw_list,sizes))
    return avg

def evaluate(model, ds, bs, dev):
    loader = DataLoader(ds, batch_size=bs*2, shuffle=False, num_workers=2, pin_memory=True)
    model.eval(); preds,labs,probs=[],[],[]
    with torch.no_grad():
        for imgs, lbls in loader:
            out = model(imgs.to(dev))
            probs += torch.softmax(out,1)[:,1].cpu().tolist()
            preds += out.argmax(1).cpu().tolist()
            labs  += lbls.tolist()
    return {
        'acc' : round(accuracy_score(labs,preds)*100,2),
        'f1'  : round(f1_score(labs,preds,zero_division=0)*100,2),
        'auc' : round(roc_auc_score(labs,probs),4),
        'prec': round(precision_score(labs,preds,zero_division=0)*100,2),
        'rec' : round(recall_score(labs,preds,zero_division=0)*100,2),
    }

total_params = sum(p.numel() for p in LightweightCNN().parameters())
print(f'Model size : {total_params:,} params  ({total_params*4/1024:.1f} KB)')

In [ ]:
# CELL 7 — Run federated training
client_idx   = partition_noniid(train_ds, NUM_CLIENTS)
global_model = LightweightCNN().to(DEVICE)
history      = []

print(f'Clients:{NUM_CLIENTS}  Rounds:{NUM_ROUNDS}  LocalEpochs:{LOCAL_EPOCHS}  Batch:{BATCH_SIZE}')
print('='*68)

for r in range(1, NUM_ROUNDS+1):
    cw, cs = [], []
    for c in range(NUM_CLIENTS):
        w, n = local_train(global_model, train_ds, client_idx[c],
                           LOCAL_EPOCHS, LR, BATCH_SIZE, DEVICE)
        cw.append(w); cs.append(n)
    global_model.load_state_dict(fed_avg(cw, cs))
    m = evaluate(global_model, test_ds, BATCH_SIZE, DEVICE)
    history.append(m)
    print(f"Rd {r:2d} | Acc:{m['acc']:5.2f}% | F1:{m['f1']:5.2f}% | AUC:{m['auc']:.4f} | Prec:{m['prec']:5.2f}% | Rec:{m['rec']:5.2f}%")

print('='*68)
final = history[-1]
print(f"FINAL → Accuracy:{final['acc']}%  F1:{final['f1']}%  AUC:{final['auc']}  Precision:{final['prec']}%  Recall:{final['rec']}%")

In [ ]:
# CELL 8 — Save to Drive and download
import json, shutil

torch.save(global_model.state_dict(), '/content/model.pth')
with open('/content/class_names.json','w') as f:
    json.dump(CLASS_NAMES, f)

# Backup to Drive
save_dir = '/content/drive/MyDrive/leukemia_model'
os.makedirs(save_dir, exist_ok=True)
shutil.copy('/content/model.pth',        f'{save_dir}/model.pth')
shutil.copy('/content/class_names.json', f'{save_dir}/class_names.json')
print(f'Backed up to Drive: {save_dir}')

# Download to your PC
from google.colab import files
files.download('/content/model.pth')
files.download('/content/class_names.json')
print('Download started — put both files in the model/ folder of your web app.')